# Scatter and paired plots

Neurotransmitter separation, taken out of `Heatmappotofu_1`.

In [ ]:
import os
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import NLProcessing
from collections import Counter
from datetime import datetime
from matplotlib import font_manager
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

date = datetime.today().strftime('%Y%m%d')

In [ ]:
filedir = "\\Data Compilation\\Climbing_New\\"
laptop = "C:\\Users\\lnico\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee"
homecomp = "D:\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee"   # TODO: update when home Dropbox moves to NUS Dropbox
labcomp = "C:\\Users\\User\\NUS Dropbox\\acclab\\Nicole M Lee"
specifiedpath = labcomp

openPath = specifiedpath + filedir
newfile2 = openPath + "Compilation with delta\\2025deltagcollection\\"

for f in font_manager.findSystemFonts(fontpaths=["fonts"]):
    font_manager.fontManager.addfont(f)
plt.rcParams["font.family"] = "Inter"
matplotlib.rcParams['svg.fonttype'] = 'none'

## Load

In [ ]:
files2 = os.listdir(newfile2)
totalfile = pd.DataFrame()

for n in files2:
    openfile = pd.read_csv(newfile2 + n)
    openfile['responder'] = n.split(" x ")[1].split("_")[0]
    totalfile = pd.concat([totalfile, openfile], axis=0).reset_index(drop=True)

totalfile['genotypeandresponder'] = totalfile["MBON"] + "_" + totalfile['responder']

onlymeans = totalfile.loc[:, ~totalfile.columns.str.endswith('_bootstrap')].drop_duplicates().reset_index(drop=True)
mbononly = onlymeans[~onlymeans['MBON'].isin(['R58', 'Th-Gal4', 'R76B09', 'VT999036'])]

cols_only = ['MBON', 'responder','genotypeandresponder', 'height_deltag', 'speed_deltag',
       'bspeed_deltag', 'maxvelocity_deltag',
       'straightindex_deltag', 'meanbout_deltag', 'bout_deltag']

RENAME = {'speed': 'Speed', 'bspeed': 'Bout speed', 'pausepos': 'Pause position', 'bout': '# Bouts', 'meanbout': "Mean bout time",
          'straightindex': 'Straightness Index', 'height': 'Avg height', 'maxvelocity': 'Max velocity'}

## Lobe location

In [ ]:
MBONList = list(set([yy.split(" ")[0] for yy in files2]))

csvfile = pd.read_csv(specifiedpath + "\\Data Compilation\\MBONlist.csv").astype('string')
for n, k in zip(["B", "y", "a"], ['\u03b2', "\u03b3", "\u03b1"]):
    csvfile['Lobe'] = csvfile['Lobe'].str.replace(n, k)

lobelocation = pd.DataFrame()
for m in MBONList:
    lobeloc = pd.DataFrame()
    lobeloc['MBON'] = [m]
    lobeloc['Lobe_location'] = [NLProcessing.find_number(csvfile, m, "Lobe")]
    lobeloc['MBON number'] = [NLProcessing.find_number(csvfile, m, "MBON number").strip()]
    lobeloc['Neurotransmitter'] = [NLProcessing.find_number(csvfile, m, "Neurotransmitter")]
    lobelocation = pd.concat([lobelocation, lobeloc])

lobelocation = lobelocation.reset_index(drop=True)


def addlobes(df):
    lobloclst = []
    mbonloclst = []
    out = df.copy()
    for n in df['MBON']:
        lobloclst.append(lobelocation[lobelocation['MBON'] == n]['Lobe_location'].values[0])
        mbonloclst.append(lobelocation[lobelocation['MBON'] == n]['MBON number'].values[0])
    out['Lobe'] = lobloclst
    out['Name'] = mbonloclst
    return out.sort_values(by = "Name", ascending=True).reset_index(drop=True)

## Select responder

In [ ]:
responder = "ACR"          # ACR or Chrimson2
responderrename = "CsChrimson" if responder == "Chrimson2" else "GtACR1"

dfoneresponder = mbononly[cols_only]
df_specificmbon = dfoneresponder[dfoneresponder['responder']==responder].reset_index(drop=True)
df_specificmbon.columns = df_specificmbon.columns.str.replace('_deltag', '')

dfreglobe = df_specificmbon.merge(lobelocation, on = "MBON", how = "inner")
df00 = dfreglobe.copy()
novt = df00[(df00['MBON']== "VT999036")].index
df50 = df00.drop(novt)
df50 = df50.drop(['MBON', 'responder', 'genotypeandresponder', 'Lobe_location', "MBON number"], axis =1)

## Scatter plot

In [ ]:
fig, axes = plt.subplots(figsize=(16,4))

df89 = pd.melt(df50, id_vars = ['Neurotransmitter'], var_name = 'Metrics', value_name = "\u0394g")
g1 = sns.swarmplot(data=df89, x= 'Metrics', y = "\u0394g", hue='Neurotransmitter', dodge = True)
sns.move_legend(g1, "upper left", bbox_to_anchor=(1, 1))
sns.set_style("darkgrid")
NLProcessing.wrap_labels(axes, 10)
g1.set_title('Plot of MBONs > ' + responderrename + ' and their \u0394g separated by neurotransmitter types according to locomotor metrics', weight='bold', fontsize =12 )

plt.savefig(openPath + "images\\" + date + "_" + responder + "_separationbyNTtypes.svg", bbox_inches='tight')

## Paired plot

In [ ]:
t1 = sns.pairplot(data=df50, hue='Neurotransmitter', corner =False, hue_order=["Glutamate", "Acetylcholine", "GABA"])
t1.fig.suptitle('MBONs > ' + responderrename, weight='bold', fontsize =16, y = 0.99 )

handles = t1._legend_data.values()
labels = t1._legend_data.keys()
t1.fig.legend(handles=handles, labels=labels, loc='lower center', ncol=5)

plt.savefig(openPath + "images\\" + date + "_" + responder + "_pairplot.svg", bbox_inches='tight')